# 1. Imports

In [2]:
import pandas as pd
import numpy as np

# 2. Load Raw Data

In [3]:
df = pd.read_csv("../data/raw/KaggleV2-May-2016.csv")
df.shape

(110527, 14)

# 3. Fix Invalid Age Values

In [4]:
df = df[df['Age'] >= 0]
df['Age'].describe()

count    110526.000000
mean         37.089219
std          23.110026
min           0.000000
25%          18.000000
50%          37.000000
75%          55.000000
max         115.000000
Name: Age, dtype: float64

# 4. Convert Date Columns to Datetime

In [5]:
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df.dtypes

PatientId                     float64
AppointmentID                   int64
Gender                         object
ScheduledDay      datetime64[ns, UTC]
AppointmentDay    datetime64[ns, UTC]
Age                             int64
Neighbourhood                  object
Scholarship                     int64
Hipertension                    int64
Diabetes                        int64
Alcoholism                      int64
Handcap                         int64
SMS_received                    int64
No-show                        object
dtype: object

# 5. Fix Negative WaitingDays Issue

In [6]:
df['WaitingDays'] = (df['AppointmentDay'].dt.normalize() - df['ScheduledDay'].dt.normalize()).dt.days
df['WaitingDays'].describe()

count    110526.000000
mean         10.183794
std          15.255034
min          -6.000000
25%           0.000000
50%           4.000000
75%          15.000000
max         179.000000
Name: WaitingDays, dtype: float64

In [7]:
df[df['WaitingDays'] < 0].shape

(5, 15)

# 6. Remove Invalid WaitingDays Rows

In [8]:
df = df[df['WaitingDays'] >= 0]
df['WaitingDays'].describe()

count    110521.000000
mean         10.184345
std          15.255153
min           0.000000
25%           0.000000
50%           4.000000
75%          15.000000
max         179.000000
Name: WaitingDays, dtype: float64

# 7. Handle Duplicate AppointmentID

In [9]:
df['AppointmentID'].duplicated().sum()

np.int64(0)

# 8. Encode Target Variable

In [10]:
df['No-show'] = df['No-show'].map({'No': 0, 'Yes': 1})
df['No-show'].value_counts()

No-show
0    88207
1    22314
Name: count, dtype: int64

# 9. Encode Gender Column

In [11]:
df['Gender'] = df['Gender'].map({'F': 0, 'M': 1})
df['Gender'].value_counts()

Gender
0    71836
1    38685
Name: count, dtype: int64

# 10. Handle Handcap Column (Convert to Binary Flag)

In [12]:
df['Handcap'].value_counts()

Handcap
0    108282
1      2040
2       183
3        13
4         3
Name: count, dtype: int64

In [13]:
df['Handcap'] = df['Handcap'].apply(lambda x: 1 if x > 0 else 0)
df['Handcap'].value_counts()

Handcap
0    108282
1      2239
Name: count, dtype: int64

# 11. Neighbourhood Encoding Strategy

In [14]:
neighbourhood_freq = df['Neighbourhood'].value_counts()
df['Neighbourhood_Freq'] = df['Neighbourhood'].map(neighbourhood_freq)
df[['Neighbourhood', 'Neighbourhood_Freq']].head()

,Neighbourhood,Neighbourhood_Freq
0,JARDIM DA PENHA,3877
1,JARDIM DA PENHA,3877
2,MATA DA PRAIA,644
3,PONTAL DE CAMBURI,69
4,JARDIM DA PENHA,3877


# 12. Drop Original Neighbourhood Column

In [15]:
df = df.drop(columns=['Neighbourhood'])
df.columns

Index(['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay',
       'AppointmentDay', 'Age', 'Scholarship', 'Hipertension', 'Diabetes',
       'Alcoholism', 'Handcap', 'SMS_received', 'No-show', 'WaitingDays',
       'Neighbourhood_Freq'],
      dtype='object')

# 13. Feature Engineering — Appointment Weekday

In [16]:
df['AppointmentWeekday'] = df['AppointmentDay'].dt.day_name()
df['AppointmentWeekday'].value_counts()

AppointmentWeekday
Wednesday    25866
Tuesday      25638
Monday       22713
Friday       19019
Thursday     17246
Saturday        39
Name: count, dtype: int64

# 14. Feature Engineering — Appointment Month

In [18]:
df['AppointmentMonth'] = df['AppointmentDay'].dt.month
df['AppointmentMonth'].value_counts()

AppointmentMonth
5    80836
6    26450
4     3235
Name: count, dtype: int64

# 15. Feature Engineering — Age Group

In [19]:
bins = [0, 12, 18, 35, 60, 115]
labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, include_lowest=True)
df['AgeGroup'].value_counts()

AgeGroup
Adult          37760
Young Adult    24135
Child          21035
Senior         19761
Teen            7830
Name: count, dtype: int64

# 16. Feature Engineering — Chronic Disease Count

In [20]:
df['ChronicDiseaseCount'] = df['Hipertension'] + df['Diabetes'] + df['Alcoholism']
df['ChronicDiseaseCount'].value_counts()

ChronicDiseaseCount
0    85306
1    17582
2     7377
3      256
Name: count, dtype: int64

# 17. Feature Engineering — Previous Attendance Rate (Per Patient)

In [21]:
df = df.sort_values(by=['PatientId', 'ScheduledDay']).reset_index(drop=True)
df[['PatientId', 'ScheduledDay', 'No-show']].head(10)

,PatientId,ScheduledDay,No-show
0,3.921784e+04,2016-05-31 10:56:41+00:00,0
1,4.374176e+04,2016-06-01 14:22:58+00:00,0
2,9.377953e+04,2016-05-18 09:12:29+00:00,0
3,1.417242e+05,2016-04-29 07:13:36+00:00,0
4,5.376153e+05,2016-04-29 07:19:57+00:00,0
5,5.628261e+06,2016-05-10 11:58:18+00:00,1
6,1.183186e+07,2016-05-19 09:42:07+00:00,0
7,2.263866e+07,2016-04-14 07:23:30+00:00,0
8,2.263866e+07,2016-05-18 13:37:12+00:00,0
9,5.216894e+07,2016-04-20 11:22:15+00:00,0


In [22]:
df['PreviousAttendanceRate'] = (
    df.groupby('PatientId')['No-show']
    .transform(lambda x: x.shift().expanding().mean())
)
df[['PatientId', 'ScheduledDay', 'No-show', 'PreviousAttendanceRate']].head(10)

,PatientId,ScheduledDay,No-show,PreviousAttendanceRate
0,3.921784e+04,2016-05-31 10:56:41+00:00,0,NaN
1,4.374176e+04,2016-06-01 14:22:58+00:00,0,NaN
2,9.377953e+04,2016-05-18 09:12:29+00:00,0,NaN
3,1.417242e+05,2016-04-29 07:13:36+00:00,0,NaN
4,5.376153e+05,2016-04-29 07:19:57+00:00,0,NaN
5,5.628261e+06,2016-05-10 11:58:18+00:00,1,NaN
6,1.183186e+07,2016-05-19 09:42:07+00:00,0,NaN
7,2.263866e+07,2016-04-14 07:23:30+00:00,0,NaN
8,2.263866e+07,2016-05-18 13:37:12+00:00,0,0.0
9,5.216894e+07,2016-04-20 11:22:15+00:00,0,NaN


# 18. Fill Missing PreviousAttendanceRate (First-time Patients)

In [23]:
overall_noshow_rate = df['No-show'].mean()
df['PreviousAttendanceRate'] = df['PreviousAttendanceRate'].fillna(overall_noshow_rate)
df['PreviousAttendanceRate'].describe()

count    110521.000000
mean          0.214417
std           0.241531
min           0.000000
25%           0.000000
50%           0.201898
75%           0.201898
max           1.000000
Name: PreviousAttendanceRate, dtype: float64

# 19. Feature Engineering — Risk History (Cumulative Past No-shows Count)

In [24]:
df['RiskHistory'] = (
    df.groupby('PatientId')['No-show']
    .transform(lambda x: x.shift().expanding().sum())
)
df['RiskHistory'] = df['RiskHistory'].fillna(0)
df['RiskHistory'].describe()

count    110521.000000
mean          0.229947
std           0.677812
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          17.000000
Name: RiskHistory, dtype: float64

# 20. Feature Engineering — Reminder Effectiveness Score

In [25]:
df['ReminderEffectivenessScore'] = df['SMS_received'] * (1 - df['PreviousAttendanceRate'])
df['ReminderEffectivenessScore'].describe()

count    110521.000000
mean          0.252840
std           0.388805
min           0.000000
25%           0.000000
50%           0.000000
75%           0.798102
max           1.000000
Name: ReminderEffectivenessScore, dtype: float64

# 21. Final Column Check

In [26]:
df.columns

Index(['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay',
       'AppointmentDay', 'Age', 'Scholarship', 'Hipertension', 'Diabetes',
       'Alcoholism', 'Handcap', 'SMS_received', 'No-show', 'WaitingDays',
       'Neighbourhood_Freq', 'AppointmentWeekday', 'AppointmentMonth',
       'AgeGroup', 'ChronicDiseaseCount', 'PreviousAttendanceRate',
       'RiskHistory', 'ReminderEffectivenessScore'],
      dtype='object')

# 22. Drop Identifier and Raw Date Columns

In [27]:
df = df.drop(columns=['PatientId', 'AppointmentID', 'ScheduledDay', 'AppointmentDay'])
df.columns

Index(['Gender', 'Age', 'Scholarship', 'Hipertension', 'Diabetes',
       'Alcoholism', 'Handcap', 'SMS_received', 'No-show', 'WaitingDays',
       'Neighbourhood_Freq', 'AppointmentWeekday', 'AppointmentMonth',
       'AgeGroup', 'ChronicDiseaseCount', 'PreviousAttendanceRate',
       'RiskHistory', 'ReminderEffectivenessScore'],
      dtype='object')

# 23. Encode Categorical Columns (AppointmentWeekday, AgeGroup)

In [28]:
df = pd.get_dummies(df, columns=['AppointmentWeekday', 'AgeGroup'], drop_first=True)
df.columns

Index(['Gender', 'Age', 'Scholarship', 'Hipertension', 'Diabetes',
       'Alcoholism', 'Handcap', 'SMS_received', 'No-show', 'WaitingDays',
       'Neighbourhood_Freq', 'AppointmentMonth', 'ChronicDiseaseCount',
       'PreviousAttendanceRate', 'RiskHistory', 'ReminderEffectivenessScore',
       'AppointmentWeekday_Monday', 'AppointmentWeekday_Saturday',
       'AppointmentWeekday_Thursday', 'AppointmentWeekday_Tuesday',
       'AppointmentWeekday_Wednesday', 'AgeGroup_Teen', 'AgeGroup_Young Adult',
       'AgeGroup_Adult', 'AgeGroup_Senior'],
      dtype='object')

# 24. Train-Test Split

In [29]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['No-show'])
y = df['No-show']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape

((88416, 24), (22105, 24))

# 25. Save Processed Data

In [30]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)